## **Chemical Equilibrium - 1 - A Chemical Thermodynamics Perspective**

<div align="left">
  <table border="1" cellpadding="6" cellspacing="0">
    <tr>
      <td bgcolor="#444444">
        <font color="#ffeb3b"><tt><b>Last updated (YYYY-MM-DD): 2026-01-08</b></tt></font>
      </td>
    </tr>
  </table>
</div>



### **Preparing the Computational Environment**  

*The next two cells prepare the environment by installing the necessary packages, importing libraries, and loading constants and functions. Make sure to run both before starting part (a) or part (b) of this notebook.*

⏳ **Note:** This may take **a few minutes** to complete. ⏳  


In [ ]:
#@title <small><small> { display-mode: "form" }
%%capture captured_output

# --- system (for LaTeX text in matplotlib) --- #
! sudo apt update -y
! sudo apt install -y cm-super dvipng texlive-latex-extra texlive-latex-recommended


In [ ]:
#@title <small><small> { display-mode: "form" }

# ---- Import some libraries of interest ----
import numpy                as np
import matplotlib.pyplot    as plt
import matplotlib.ticker    as mticker
import ipywidgets           as w
import plotly.graph_objects as go
# - - - - - - - - - - - - - - - - - - - - - -
from IPython.display import display
from google.colab    import files
from google.colab    import output
from plotly.subplots import make_subplots
from scipy.optimize  import root_scalar
# -------------------------------------------

# ---- Define some constants of interest ----
R        = 8.31447 # in S.I. (J / K / mol)
P_o      = 1E5     # standard pressure (P^o = 1E5 Pa = 1 bar)
ZERO     = 1e-300  # |x|<ZERO --> x = ZERO
ZERO2    = 1E-7
# -------------------------------------------
last_fig = None
# -------------------------------------------
FONTSIZE = [11,12,14,15,16,20]
# -------------------------------------------

# ============================================== #
# For all the following functions, it is assumed #
# that all data are provided in SI units.        #
# Moreover, P is used for total pressure,        #
# whereas lower p is used for partial pressures. #
# ============================================== #

# ---- Get reaction from the string given by user ----
def string_to_reaction(string):
    reactants, products = string.split("->")
    reactants = [i.strip() for i in reactants.split("+")]
    products  = [i.strip() for i in  products.split("+")]
    nus,molecules = [],[]
    for r in reactants:
        r = [i.strip() for i in r.split()]
        if   len(r) == 1: nu,molecule = 1,r[0]
        elif len(r) == 2: nu,molecule = r
        else            : raise Exception
        nus.append(-int(nu))
        molecules.append(molecule)
    for r in products:
        r = [i.strip() for i in r.split()]
        if   len(r) == 1: nu,molecule = 1,r[0]
        elif len(r) == 2: nu,molecule = r
        else            : raise Exception
        nus.append(+int(nu))
        molecules.append(molecule)
    return np.array(nus),np.array(molecules)

# ---- Write the equation of the reaction ----
def reaction_to_string(nus,molecules):
    stringR = []
    stringP = []
    for nu, molecule in zip(nus,molecules):
        if nu < 0: stringR += ["%s * %s"%(-nu,molecule)]
        if nu > 0: stringP += ["%s * %s"%(+nu,molecule)]
    stringR = " + ".join(stringR)
    stringP = " + ".join(stringP)
    sreaction = rf"{stringR:s} ⇌ {stringP:s}"
    return str(sreaction)

# ---- Limits for extent of reaction ----
def limits_xi(n_0,nus):

    # maximum value of xi (calculated considering consumption of reactants)
    xi_max = min([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i < 0])

    # minimum value of xi (calculated considering consumption of products)
    xi_min = max([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i > 0])

    # Ensure numerical zeros are displayed as +0.0 for clarity
    if xi_min == -0.0: xi_min = 0.0
    if xi_max == -0.0: xi_max = 0.0

    return xi_min,xi_max

# ---- auxiliary function ----
def prepare_variables(T, P, nus, n_0, xi):

    T   = np.asarray(T)
    P   = np.asarray(P)
    nus = np.asarray(nus) #, dtype=float)  # (S,)
    n_0 = np.asarray(n_0) #, dtype=float)  # (S,)
    xi  = np.asarray(xi)

    # Expand for broadcasting with xi (which can be a scalar or have shape (M, N))
    # If xi.ndim == 0 (scalar), this keeps the shape (S,) -> (S,)
    # If xi.ndim == 2, it becomes (S, 1, 1) and broadcasts to (S, M, N)
    expand = (None,)*xi.ndim
    n_0_e  = n_0[(...,) + expand]
    nus_e  = nus[(...,) + expand]

    # total sums (scalars)
    dnu    = nus.sum()
    ntot_0 = n_0.sum()

    return T, P, nus_e, n_0_e, xi, dnu, ntot_0

# ---- Delta{H}^o as a function of T ----
def get_DHo(T,refdata):
    DHo_ref,DSo_ref,DCPo_ref,T_ref = refdata
    DHo_T = DHo_ref + DCPo_ref * T * (1 - T_ref/T)
    return DHo_T

# ---- Delta{G}^o as a function of T ----
def get_DGo(T,refdata):
    DHo_ref,DSo_ref,DCPo_ref,T_ref = refdata
    DGo_T  = DHo_ref - T * DSo_ref
    DGo_T += DCPo_ref  * ( T - T_ref + T*np.log(T_ref / T))
    return DGo_T

# ---- DeltaG* ----
def get_Gast(T,P,nus,refdata):
    DGast = get_DGo(T,refdata) + R*T*np.log(P/P_o) * nus.sum()
    return DGast

# ---- DeltaGmix ----
def get_DDGmix(xi, T, P, n_0, nus):
    """
    Accepts xi as either a scalar or an array (e.g., a mesh of shape (M, N)); T and p can be scalars or broadcast-compatible arrays.
    """

    T, P, nus, n_0, xi, dnu, ntot_0 = prepare_variables(T, P, nus, n_0, xi)

    # magnitudes dependent on xi
    n_xi    = n_0    + xi * nus   # (S,) o (S,M,N)
    ntot_xi = ntot_0 + xi * dnu   # scalar or (M,N)

    # Get molar fractions
    y_xi = n_xi /ntot_xi
    y_0  = n_0  /ntot_0

    # Notice that 0*ln(0) --> 0
    maskA = (n_xi > ZERO) & (y_xi > ZERO)
    termA = np.zeros_like(y_xi, dtype=float)
    termA[maskA] = n_xi[maskA] * np.log(y_xi[maskA])

    maskB = (n_0  > ZERO) & (y_0  > ZERO)
    termB = np.zeros_like(y_0, dtype=float)
    termB[maskB] = n_0[maskB]  * np.log(y_0[maskB])

    DDGmix = R*T*np.sum(termA-termB, axis=0)
    return DDGmix

#==========================================#
# ----           Constant T,P           ----
#==========================================#

# ---- Gibbs free energy for a given T,p ----
def get_G_TP(xi,n_0,nus,P,T,refdata):
    Gast   = get_Gast(T,P,nus,refdata)
    DDGmix = get_DDGmix(xi,T,P,n_0,nus)
    DGtot  = Gast * xi + DDGmix
    return DGtot

# ---- Quotient of reaction (pressure) ----
def get_Qp_TP(n_0,nus,xi,P):

    # Get partial pressures for this value of xi
    n_xi = n_0 + xi*nus
    y_xi = n_xi/n_xi.sum()
    p_xi = y_xi * P
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
         Qp = np.prod([(p_xi_j/P_o)**nu_j for nu_j,p_xi_j in zip(nus,p_xi)])
    return Qp

# ---- Extent of reaction in terms of P and T ----
def get_xieq_TP(P,T,n_0,nus,refdata):

    # Get Eq constant
    dGo_T = get_DGo(T,refdata)
    Kp_T  = np.exp(-dGo_T/R/T)

    # Get limits for xi
    xi_min,xi_max = limits_xi(n_0,nus)
    xi_guess      = 0.5 * (xi_max + xi_min)
    result        = root_scalar(lambda xi: get_Qp_TP(n_0,nus,xi,P)-Kp_T,x0=xi_guess,bracket=(xi_min,xi_max),xtol=1E-8)
    xi_eq         = float(result.root)
    return xi_eq

# ---- Extent of reaction in terms of p and T (for A --> 2B) ----
def xi_TP_N2O4(P,T,refdata):
    dG0 = get_DGo(T,refdata)
    k_p = np.exp(-dG0/R/T)
    xi  = np.sqrt(k_p / (k_p + 4 * P/P_o))
    return xi


#==========================================#
# ----           Constant T,V           ----
#==========================================#

# ---- Helmholtz free energy for a given T,V ----
def get_A_TV(xis,T,V,n_0,nus,refdata):

    ntot_0  = n_0.sum()
    ntot_xi = ntot_0 + nus.sum() * xis
    DGo_T   = get_DGo(T,refdata)

    P_xi    = ntot_xi*R*T/V

    term_lineal = DGo_T * xis
    termA   = ntot_xi * R *T * (np.log( P_xi/P_o                  )-1)
    termA  -= ntot_0  * R *T * (np.log((P_xi/P_o)*(ntot_0/ntot_xi))-1)
    DDGmix  = get_DDGmix(xis,T,P_xi,n_0,nus)
    term_nolin  = termA+DDGmix

    DAtot   = term_lineal + term_nolin

    return DAtot, term_lineal, term_nolin, P_xi

# ---- Quotient of reaction (volume) ----
def get_Qp_TV(n_0,nus,xi,V,T):

    # Get partial pressures for this value of xi

    n_xi = n_0 + xi*nus
    y_xi = n_xi/(n_xi.sum())
    P    = n_xi.sum() *R*T/V
    p_xi = y_xi * P
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
         Qp = np.prod([(p_xi_j/P_o)**nu_j for nu_j,p_xi_j in zip(nus,p_xi)])
    return Qp

# ---- Extent of reaction in terms of V and T ----
def get_xieq_TV(V,T,n_0,nus,refdata):

    # Get Eq constant
    dGo_T = get_DGo(T,refdata)
    Kp_T  = np.exp(-dGo_T/R/T)

    # Get limits for xi
    xi_min,xi_max = limits_xi(n_0,nus)
    xi_guess      = 0.5 * (xi_max + xi_min)
    result        = root_scalar(lambda xi: get_Qp_TV(n_0,nus,xi,V,T)-Kp_T,x0=xi_guess,bracket=(xi_min,xi_max))
    xi_eq         = float(result.root)
    return xi_eq


#=====================================#
# ---- FUNCTIONS FOR DOWNLOADING ---- #
#=====================================#
# ---- button to download file ----
def download_file(fname):
    btn = w.Button(
            description=f"Download {fname}",
            icon="download",
            button_style="primary",
            layout=w.Layout(width='250px', height='38px', margin='0 0 0 55px')
    )
    # action when clicking
    btn.on_click(lambda _: files.download(fname))
    return btn

def _on_download_clicked(_,conditions):
    # --- get figure from global variable ---
    fig = last_fig
    if fig is None:
        print("No figure yet; move a slider to generate the plot...")
        return
    if   conditions == "TP": fname = f"equilibrium_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif conditions == "TV": fname = f"equilibrium_T{T_slider.value:.0f}K_V{V_slider.value:.2f}L.svg"
    else                   : fname = f"equilibrium_T{T_slider.value:.0f}K.svg"

    fig.savefig(fname, bbox_inches='tight')
    files.download(fname)

#==================================#
# ---- FUNCTIONS FOR PLOTTING ---- #
#==================================#
def plot_DG_T(T,T_ref,refdata):
    # Calculation of DG at Tref
    DGo_ref    = get_DGo(T_ref,refdata)

    # Calculation of DG at other temperatures
    DGo_T = get_DGo(T,refdata)
    plt.rcParams['text.usetex'] = True

    # Plot results
    labeldot = r"$\Delta_{r}{G}^\circ(T_{\rm ref}) = %.2f \;\; \mathrm{kJ/mol}$"%(DGo_ref/1000)
    plt.plot(T    ,DGo_T  /1000,'k-',zorder=1)
    plt.plot(T_ref,DGo_ref/1000,'ro',zorder=3,label=labeldot)

    # Format plot
    plt.xticks(fontsize=FONTSIZE[4])
    plt.yticks(fontsize=FONTSIZE[4])

    plt.xlabel(r'$T \;\; (\mathrm{K})$'                        ,fontsize=FONTSIZE[5])
    plt.ylabel(r'$\Delta_{r} G^{\circ} \;\; (\mathrm{kJ/mol})$',fontsize=FONTSIZE[5])

    plt.legend(loc="best",fontsize=FONTSIZE[3])

    # ---- download button ----
    fname = 'plot_DGo_T.svg'
    fig   = plt.gcf()
    fig.savefig(fname, bbox_inches='tight')
    btn   = download_file(fname)

    # ---- show plot and close ----
    display(btn)
    plt.show()
    plt.close()

    return DGo_T
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def plot_gibbshelmholtz(T,DGo_T,refdata):
    # --- DH^o at each temperature ---
    DHo_T = get_DHo(T,refdata)
    yy    = DHo_T / (R * T**2)

    # Plot results
    fig, ax = plt.subplots()
    ax.plot(T,yy,'k-',zorder=1,label=r'$(a) \;\; f=\Delta_r H^{\circ}/T^2$ \quad\quad\quad\quad [analytic]')

    # Format plot
    ax.tick_params(axis='x', labelsize=FONTSIZE[4])
    ax.tick_params(axis='y', labelsize=FONTSIZE[4])

    ax.set_xlabel(r'$T \;\; (\mathrm{K})$'       , fontsize=FONTSIZE[5])
    ax.set_ylabel(r'$f/R \;\; (\mathrm{K}^{-1})$', fontsize=FONTSIZE[5])

    # --- Calculate numerical derivative ---
    numslope = np.gradient(-DGo_T / (R*T) , T)

    # --- Add data to plot (removing first and last points, due to numerical error) ---
    ax.plot(T[1:-1],numslope[1:-1],'xr',zorder=2, markeredgewidth=2.5,label=r'$(b)  \;\; f = -d/dT(\Delta_r G^{\circ}/T)$ \;\, [numeric]')

    ax.legend(loc="best",fontsize=FONTSIZE[3])

    # ---- download button ----
    fname = "plot_gibbshelmholtz.svg"
    fig   = plt.gcf()
    fig.savefig(fname, bbox_inches='tight')
    btn   = download_file(fname)

    # Show plot, button and close
    display(btn)
    display(fig)
    plt.close()
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def print_info_eq(magnitude,PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P_eq,Kp_v1,Kp_v2,Ky_v1,Ky_v2):

    P_0 ,V_0 ,T_0  = PVT_0
    P_eq,V_eq,T_eq = PVT_eq

    sP_0  = rf"{P_0/1E5:.2f}"
    sP_eq = rf"{P_eq/1E5:.2f}"
    Pformat = max(len(sP_0),len(sP_eq))

    sV_0  = rf"{V_0*1E3:.2f}"
    sV_eq = rf"{V_eq*1E3:.2f}"
    Vformat = max(len(sV_0),len(sV_eq))

    print("")
    print(fr"Initial  conditions: ({T_0:.2f}K,{sP_0:{Pformat:d}s}bar,{sV_0:{Vformat:d}s}L)")
    print("")
    print(fr"Equilib. conditions: ({T_eq:.2f}K,{sP_eq:{Pformat:d}s}bar,{sV_eq:{Vformat:d}s}L)")
    print("")
    print(fr"   ==> equilibrium found at xi_eq = {xi_eq:.4f} mol")
    print("")

    print(fr"   Number of moles & molar fraction at equilibrium (from xi_eq):")
    for j,molecule in enumerate(molecules):
        n_eq_j = n_eq[j]
        y_eq_j = y_eq[j]
        print(fr"   * n_i = {n_eq_j:6.4f} mol, y_i = {y_eq_j:7.4f} (i = {molecule:s})")
    print("")

    print(fr"   Partial pressures at equilibrium:")
    for j,molecule in enumerate(molecules):
        p_eq_j = p_eq[j]/1E5
        print(fr"   * p_i = {p_eq_j:7.4f} bar (i = {molecule:s})")
    print("")


    if 9999 > Kp_v1 > 0.100: sformat1 = ".3f"
    else                   : sformat1 = ".2E"
    if 9999 > Ky_v1 > 0.100: sformat2 = ".3f"
    else                   : sformat2 = ".2E"

    reldiff = 100*abs(Kp_v2-Kp_v1)/Kp_v1
    if reldiff < 5.0:
        print(fr"   Value of Kp:")
        print(fr"   * from Delta_r{{G}}^o --> Kp = {Kp_v2:{sformat1}} [*]")
        print(fr"   * from p_i values   --> Kp = {Kp_v1:{sformat1}}")
        print("")
        print(fr"   Value of Ky:")
        if Ky_v2 is not None:
          print(fr"   * from Delta_r{{G}}^* --> Ky = {Ky_v2:{sformat2}} [*]")
        print(fr"   * from y_i values   --> Ky = {Ky_v1:{sformat2}}")
        print("")
        print(fr"   The previous values for the equilibrium constants may vary slightly")
        print(fr"   due to numerical errors. Trust the values with the [*].")
    else:
        print(fr"   Value of Kp:")
        print(fr"   * from Delta_r{{G}}^o --> Kp = {Kp_v2:{sformat1}}")
        print("")
        if Ky_v2 is not None:
          print(fr"   Value of Ky:")
          print(fr"   * from Delta_r{{G}}^* --> Ky = {Ky_v2:{sformat2}}")
    print("")
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def plot_DG_TP(T,P, fixed_args):

    molecules,nus,n_0,xis,refdata = fixed_args

    V_0 = n_0.sum()*R*T/P

    # ---- Calculate DG* ----
    DGo   = get_DGo(T,refdata)
    Gast  = get_Gast(T,P,nus,refdata)

    # ---- Terms in DGtot ----
    DGlin  = Gast * xis
    DDGmix = get_DDGmix(xis,T,P,n_0,nus)
    DGtot  = DGlin + DDGmix

    # ---- minimum of DGtot ---
    xi_eq  = get_xieq_TP(P,T,n_0,nus,refdata)
    minDG  = get_G_TP(xi_eq,n_0,nus,P,T,refdata)
    minDG  = minDG/(R*T)

    # ---- minimum of DGmix ---
    min2xi  = xis[np.argmin(DDGmix)]
    min2DG  = np.min(DDGmix)/(R*T)

    # ---- Number of moles and pressure at equilibrium ----
    n_eq = n_0 + xi_eq * nus
    y_eq = n_eq / n_eq.sum()
    p_eq = P * y_eq
    V_eq = (n_eq.sum())*R*T/P

    # ---- Calculation of Kp and Ky ----
    # (a) using molar fractions and partial pressures (from xi_eq)
    Kp_v1, Ky_v1 = 1.0, 1.0
    for p_eq_j,nu_j in zip(p_eq,nus): Kp_v1 *= (p_eq_j/P_o)**nu_j
    for y_eq_j,nu_j in zip(y_eq,nus): Ky_v1 *= y_eq_j**nu_j
    # (b) using deltaG values
    Ky_v2 = np.exp(-Gast/(R*T))
    Kp_v2 = np.exp(-DGo /(R*T))
    # (c) directly from xi_eq (specific case A --> 2B)
    Ky_v3 = 4*xi_eq * xi_eq / (1-xi_eq*xi_eq)

    # --- Plot data ---
    fig = plt.figure(figsize=(5.50,3.67))
    plt.plot(xis, DGlin/(R*T), color='b',ls="--",label=r"$f = \xi \cdot \Delta_{r} G^{\ast}$")
    plt.plot(xis,DDGmix/(R*T), color='r',ls=":" ,label=r"$f = \Delta_{\rm mix}G(\xi) - \Delta_{\rm mix}G(0)$")
    plt.plot(xis, DGtot/(R*T), color='k',ls="-" ,label=r"$f = G(\xi) - G(0)$")
    plt.plot( xi_eq ,minDG ,'ko' )
    plt.plot( min2xi,min2DG,'rx' )

    # some formatting
    plt.xticks(fontsize=FONTSIZE[1])
    plt.yticks(fontsize=FONTSIZE[1])
    plt.xlabel(r'$\xi \;\; \mathrm{[mol]}$'           , fontsize=FONTSIZE[2])
    plt.ylabel(r'$f \; / \; (RT) \;\; \mathrm{[mol]}$', fontsize=FONTSIZE[2])
    plt.title(fr'T={T:.0f} K, p={P/1E5:.2f} bar')

    # secondary grid in x-axis
    ax = plt.gca()
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(True, which='minor', axis='x', alpha=0.15)
    ax.tick_params(axis='x', which='minor',bottom=False,top=False,length=0)
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend(loc="best", fontsize=FONTSIZE[0])

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    # --- Show and close figure ---
    plt.show()
    plt.close()

    # ---- Print info ----
    PVT_0  = (P,V_0 ,T)
    PVT_eq = (P,V_eq,T)
    # print(fr"Value for Delta_r{{G}}^o({T:.2f} K) = {DGo/1000:.2f} kJ/mol")
    # print(fr"Value for Delta_r{{G}}^*({T:.2f} K) = {Gast/1000:.2f} kJ/mol")
    print_info_eq("G",PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P,Kp_v1,Kp_v2,Ky_v1,Ky_v2)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def plot_DA_TV(T,V, fixed_args):

    molecules,nus,n_0,xis,refdata = fixed_args

    # ---- Terms in DAtot ----
    DAtot, term_lineal, term_nolin, P_xi = get_A_TV(xis,T,V,n_0,nus,refdata)

    # ---- minimum of DAtot ---
    idx_eq = np.argmin(DAtot)
    xi_eq  = xis[idx_eq]
    P_eq   = P_xi[idx_eq]
    minDA  = np.min(DAtot)/(R*T)

    # ---- minimum of DAtot ---
    xi_eq  = get_xieq_TV(V,T,n_0,nus,refdata)
    minDA  = get_A_TV(xi_eq,T,V,n_0,nus,refdata)[0]/(R*T)

    # ---- minimum of non-lineal term ---
    min2xi  = xis[np.argmin(term_nolin)]
    min2yy  = np.min(term_nolin)/(R*T)

    # ---- Number of moles and pressure at equilibrium ----
    n_eq = n_0 + xi_eq * nus
    y_eq = n_eq / n_eq.sum()
    p_eq = P_eq * y_eq

    # ---- Calculation of Kp and Ky ----
    # (a) using molar fractions and partial pressures (from xi_eq)
    Kp_v1, Ky_v1 = 1.0, 1.0
    for p_eq_j,nu_j in zip(p_eq,nus): Kp_v1 *= (p_eq_j/P_o)**nu_j
    for y_eq_j,nu_j in zip(y_eq,nus): Ky_v1 *= y_eq_j**nu_j
    # (b) using deltaG values
    DGo          = get_DGo(T,refdata)
    Kp_v2, Ky_v2 = np.exp(-DGo /(R*T)), None

    # --- Plot data ---
    plt.figure(figsize=(5.50,3.67))

    plt.plot(xis,term_lineal/(R*T), color='b',ls="--",label=r"$f = \xi \cdot \Delta_{r} G^{o}$")
    plt.plot(xis,term_nolin /(R*T), color='r',ls=":",label=r"$f = V \sum_i \left[ p_i \ln\frac{p_i}{e p^\circ} - p_i(0) \ln \frac{p_i(0)}{e p^\circ} \right]$")
    plt.plot(xis,      DAtot/(R*T), color='k',ls="-" ,label=r"$f = A(\xi) - A(0)$")

    plt.plot( xi_eq ,minDA ,'ko' )
    plt.plot(min2xi ,min2yy ,'rx' )

    plt.xticks(fontsize=FONTSIZE[1])
    plt.yticks(fontsize=FONTSIZE[1])
    plt.xlabel(r'$\xi \;\; \mathrm{[mol]}$'           , fontsize=FONTSIZE[2])
    plt.ylabel(r'$f \; / \; (RT) \;\; \mathrm{[mol]}$', fontsize=FONTSIZE[2])
    plt.title(fr'T={T:.0f} K,  V={V*1000:.2f} L')

    # secondary grid in x-axis
    ax = plt.gca()
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(True, which='minor', axis='x', alpha=0.15)
    ax.tick_params(axis='x', which='minor',bottom=False,top=False,length=0)
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend(loc="best",fontsize=FONTSIZE[0])

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    # --- Show and close figure ---
    plt.show()
    plt.close()

    # ---- Print info ----
    P_0 = n_0.sum() * R*T/V
    PVT_0  = (P_0 ,V,T)
    PVT_eq = (P_eq,V,T)
    print_info_eq("A",PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P_eq,Kp_v1,Kp_v2,Ky_v1,Ky_v2)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#

#==================================#
# ---- Interaction with student ----
def ask_for_float(question,ntries=3):
    count = 0
    while True:
      try:
        value = float(input(question))
        break
      except:
        count += 1
        if count == ntries:
              print("      getting data failed too many times... aborting!")
              raise Exception
        print("      something went wrong... trying again...")
    return value
#==================================#

#==========================================#
# ----   Specific for N2O4 <---> 2NO2   ----
#==========================================#
def load_n2o4_2no2():
    # from Atkins' Physical Chemistry
    T_ref    = 298
    DHo_ref  =  57.20 * 1E3
    DSo_ref  = 175.83
    DCPo_ref =  -2.88
    #------------------------------------------------------
    refdata = (DHo_ref,DSo_ref,DCPo_ref,T_ref)
    #------------------------------------------------------
    molecules     = ["N2O4","NO2"]
    nus           = np.array([-1,2])
    #------------------------------------------------------
    n_0           = np.array([1.00, 0.00])
    #------------------------------------------------------
    return refdata,molecules,nus,n_0

def plot3D_eqPT_N2O4(T,P):

    refdata,molecules,nus,n_0 = load_n2o4_2no2()

    # calculate data for each plot
    Z1 = xi_TP_N2O4(P,T,refdata)
    Z2 = get_G_TP(Z1,n_0,nus,P,T,refdata)/(R*T)

    # Create figure with two subplots
    # title1 = r'$(a) \;\; z: \;\; \xi_{\mathrm{eq}} \;\; \text{(mol)}$'
    # title2 = r'$(b) \;\; z: \;\; [G(\xi_{\mathrm{eq}}) - G(0)] / RT  \;\; \text{(mol)}$'
    title1 = "(a)  z:  <i>ξ</i><sub>eq</sub> (mol)"
    title2 = "(b)  z:  [<i>G</i>(<i>ξ</i><sub>eq</sub>) − <i>G</i>(0)] / <i>RT</i>  (mol)"
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=(title1,title2))

    fig.add_trace(go.Surface(x=P/1E5, y=T, z=Z1, colorscale='Viridis', showscale=False),row=1, col=1)
    fig.add_trace(go.Surface(x=P/1E5, y=T, z=Z2, colorscale='Plasma' , showscale=False),row=1, col=2)

    # formating
    fig.update_layout(
        width=1000, height=450,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
          xaxis=dict(title=dict(text='p (bar)',font=dict(size=18))),
          yaxis=dict(title=dict(text='T (K)'  ,font=dict(size=18))),
          zaxis=dict(title=dict(text=''       ,font=dict(size=18))),
          domain=dict(x=[0.00, 0.50], y=[0, 1]),
          camera=dict(eye=dict(x=1.6, y=1.6, z=0.9))),
        scene2=dict(
          xaxis=dict(title=dict(text='p (bar)',font=dict(size=18))),
          yaxis=dict(title=dict(text='T (K)'  ,font=dict(size=18))),
          zaxis=dict(title=dict(text=''       ,font=dict(size=18))),
          domain=dict(x=[0.50, 1.00], y=[0, 1]),
          camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)))
    )

    # Ajust position of plot titles
    for ann in fig['layout']['annotations']:
        ann['y'] -= 0.15
        ann['font'] = dict(size=16)

    fig.show(config={"toImageButtonOptions": {"format": "svg","filename": "equilibrium_surfaces","scale": 2}})

def N2O4_yB_to_xi(yB):
    refdata,molecules,nus,n_0 = load_n2o4_2no2()
    xi = (yB*n_0.sum() - n_0[1])/(2-yB)
    return xi

def interceptN2O4_getGm(T,P,yB):
    refdata,molecules,nus,n_0 = load_n2o4_2no2()
    xi     = N2O4_yB_to_xi(yB)
    ntot   = n_0.sum() + xi
    Gtot   = get_G_TP(xi,n_0,nus,P,T,refdata)
    Gm     = Gtot/ntot
    return Gm, Gtot, ntot

def interceptN2O4_getline(T,P,yB_i):

    refdata,molecules,nus,n_0 = load_n2o4_2no2()

    #partial pressures and quotient of reaction
    yA_i = 1-yB_i
    pB_i = P*yB_i
    pA_i = P*yA_i
    Qp   = (pB_i/P_o)**2 / (pA_i/P_o)
    # value for xi_i
    xi_i = N2O4_yB_to_xi(yB_i)
    # delta_r G^o (T)
    dGo_T  = get_DGo(T,refdata)
    # G_tot
    Gm_i, Gtot_i, ntot_i = interceptN2O4_getGm(T,P,yB_i)

    # get slope (m)
    dxidyB = (2*n_0.sum() - n_0[1]) / (2-yB_i)**2
    dGdxi  = dGo_T + R*T*np.log(Qp)
    m      = 1/ntot_i * dxidyB * (dGdxi - Gm_i)
    # Point of the line
    xx     = yB_i
    yy     = (Gm_i - interceptN2O4_getGm(T,P,0)[0])/(R*T)
    # get intercept (b)
    m      = m / (R*T)
    b      = yy - m*xx
    return m,b, (xx,yy)

def interceptN2O4_plot(T,P,yB):

    refdata,molecules,nus,n_0 = load_n2o4_2no2()

    # equilibrium
    xi_eq   = get_xieq_TP(P,T,n_0,nus,refdata)
    yB_eq   = (n_0[1]+2*xi_eq)/(n_0.sum() + xi_eq)
    Gtot_eq = interceptN2O4_getGm(T,P,yB_eq)[1]

    # Get data in terms of yB_i
    lGm,lGtot,lntot = [],[],[]
    list_yB      = np.linspace(0.0,1.0,71)
    for yB_i in list_yB:
        Gm_i, Gtot_i, ntot_i = interceptN2O4_getGm(T,P,yB_i)
        lntot.append(ntot_i)
        lGtot.append(Gtot_i)

    # control point
    if yB == 0.0: yB = ZERO2
    if yB == 1.0: yB = 1 - ZERO2

    # equation for tangent of Gm
    m,b,(xx1,yy1) = interceptN2O4_getline(T,P,yB)

    # values at the selected point
    Gm, Gtot, ntot = interceptN2O4_getGm(T,P,yB)

    # apply reference
    Gtot    =   (Gtot    - lGtot[0])/(R*T)
    Gtot_eq =   (Gtot_eq - lGtot[0])/(R*T)
    lGtot   = [(lGtot[i] - lGtot[0])/(R*T) for i in range(len(lGtot))]
    lGm     = [lGtot[i]/lntot[i]           for i in range(len(lGtot))]

    # --- Create side-by-side axes ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

    # -- left plot --
    ax1.plot(list_yB,lGtot,'k-')
    ax1.plot(yB_eq,Gtot_eq,'ko',zorder=1)
    ax1.plot(yB   ,Gtot   ,'ro',zorder=2)

    for ii in ["x","y"]: ax1.tick_params(axis=ii,labelsize=FONTSIZE[1])
    ax1.set_xlim( 0.0 , 1.0 )
    ax1.set_xlabel(r"$y_{\rm NO_2}$",fontsize=FONTSIZE[2])
    ax1.set_ylabel(r"$[G(y_{\rm NO_2})-G(0)]\cdot (RT)^{-1} \;\; \mathrm{[mol]}$",fontsize=FONTSIZE[2])

    # -- right plot --
    xx2 = [ZERO2,1-ZERO2]
    yy2 = [m*xx_i + b for xx_i in xx2]
    ylim1 = min(lGm)
    ylim2 = max(lGm)
    delta = ylim2-ylim1
    ylim1 = ylim1 - delta*0.1
    ylim2 = ylim2 + delta*0.1

    ax2.plot(list_yB,lGm,'k-')
    ax2.plot(xx1,yy1,'ro')
    ax2.plot(xx2,yy2,'r--',label=rf"${m:+.5f} \cdot y_{{\rm NO_2}} {b:+.5f}$")

    for ii in ["x","y"]: ax2.tick_params(axis=ii,labelsize=FONTSIZE[1])
    ax2.set_xlim( 0 , 1 )
    ax2.set_ylim(ylim1,ylim2)
    ax2.set_xlabel(r"$y_{\rm NO_2}$",fontsize=FONTSIZE[2])
    ax2.set_ylabel(r"$[G_{\rm m}(y_{\rm NO_2})- G_{\rm m}(0)] \cdot (RT)^{-1}$",fontsize=FONTSIZE[2])
    ax2.legend(loc="upper center",fontsize=FONTSIZE[0])

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    # --- Show and close figure ---
    plt.show()
    plt.close()
    print(rf"Equilibrium at y(NO2) = {yB_eq:.7f}")



### **(a) The worked example: $\rm{N_2O_4(g)} \rightleftharpoons 2 \; \rm{NO_2(g)}$ reaction**

####
Let us consider, in this whole section, the following reaction:

$$\rm N_2O_4 (g) \rightleftharpoons 2\,NO_2 (g)$$

At 298 K, the following thermodynamic data are available (see Atkins' *Physical Chemistry*, **2014**, Oxford University Press):
-  for $\rm N_2O_4$ $\Rightarrow$ $\Delta_f{H}^\circ = \;\; 9.16$ kJ mol$^{-1}$, $S^\circ = 304.29$ J mol$^{-1}$ K$^{-1}$ and $C_{p,m}^\circ = 77.28$ J mol$^{-1}$ K$^{-1}$;
-  for $\rm NO_2$ $\;$ $\Rightarrow$ $\Delta_f{H}^\circ = 33.18$ kJ mol$^{-1}$, $S^\circ = 240.06$ J mol$^{-1}$ K$^{-1}$ and $C_{p,m}^\circ = 37.20$ J mol$^{-1}$ K$^{-1}$.

From these values, we obtain:
- $\Delta_{r} H^{\circ}(T_{\rm ref}) = \;\;57.20$ kJ mol$^{-1}$,
- $\Delta_{r} S^{\circ}(T_{\rm ref}) \;= 175.83$ J mol$^{-1}$ K$^{-1}$ and
- $\Delta_{r} C_p^\circ(T_{\rm ref}) \;= -2.88$ J mol$^{-1}$ K$^{-1}$.

Load this data by executing the cell below.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
refdata,molecules,nus,n_0        = load_n2o4_2no2()
DHo_ref, DSo_ref, DCPo_ref,T_ref = refdata
#------------------------------------------------------
string = reaction_to_string(nus,molecules)
print(rf"The reaction is: {string:s}")
print("")
# ---- Print values for the reaction ----
print(rf"Magnitudes of interest at Tref = {T_ref:.2f} K:")
print(rf"    Delta_r{{H}}^o  = {DHo_ref/1000:+8.2f} kJ/mol")
print(rf"    Delta_r{{S}}^o  = {DSo_ref:+8.2f}  J/(mol K)")
print(rf"    Delta_r{{Cp}}^o = {DCPo_ref:+8.2f}  J/(mol K)")
print("")

####
(a.1) EFFECT OF THE TEMPERATURE IN $\Delta_r G^\circ$

#####
For an ideal gas-phase reaction, the temperature dependence of the standard Gibbs free energy change, $\Delta_{r} G^\circ$, can be expressed as:

$$
\Delta_{r} G^{\circ}(T)= \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
 \tag{1}
$$

with $T_{\rm ref}$ being the reference $T$ and under the assumption that $\Delta_{r} C_p^\circ$ is independent of $T$.

Examine this temperature dependence of $\Delta G^\circ$ by executing the next cell. In the generated plot, the red dot indicates the value at $T_{\rm ref}$.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
refdata,molecules,nus,n_0        = load_n2o4_2no2()
DHo_ref, DSo_ref, DCPo_ref,T_ref = refdata
#------------------------------------------------------
T     = np.linspace(200,400,31)
DGo_T = plot_DG_T(T,T_ref,refdata)
#------------------------------------------------------

#####
According to the Gibbs–Helmholtz equation, the following equality should hold:

$$
\frac{\Delta_{r} H^{\circ}}{T^2} = \frac{d}{dT}\left( -\frac{\Delta_{r} G^{\circ}}{T} \right)
\tag{2}
$$

This relationship can be tested by computing the numerical derivative of the term on the right-hand side and comparing it with the analytical expression on the left-hand side.

- The analytical expression for the enthalpy term as a function of temperature is:
$$
\Delta_{r} H^{\circ} = \Delta_{r} H^{\circ}(T_{\rm ref}) + \Delta_{r} C_{p}^\circ \cdot (T- T_{\rm ref})
\tag{3}
$$
Using this expression, we can easily plot $\Delta_{r} H^{\circ}/ T^2$.

- On the other side, the numerical derivative of $-\Delta_{r} G^{\circ}/T$ with respect to temperature can be directly computed from the data obtained in the execution of the previous cell.

Execute the next cell to check the Gibbs–Helmholtz relationship.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
refdata,molecules,nus,n_0 = load_n2o4_2no2()
#------------------------------------------------------
plot_gibbshelmholtz(T,DGo_T,refdata)
#------------------------------------------------------


####
(a.2) REACTION CONDUCTED AT CONSTANT P AND T

#####
$G$ changes dynamically as the reaction progresses. By knowing the initial conditions for the reaction, it is possible to track how $G$ varies with the extent of reaction ($\xi$), which is of special interest when the reaction takes place at constant $T$ and $p$.

Let us assume that the reaction vessel initially contains 1 mol of N$_2$O$_4$ (and none of NO$_2$). Execute the cell below, set the temperature and the total pressure, and analyze how $G$ changes with $\xi$:

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
refdata,molecules,nus,n_0        = load_n2o4_2no2()
#------------------------------------------------------

#----  Get limit values for xi  ----
STEP          = 1E-4
xi_min,xi_max = limits_xi(n_0,nus)
#----     Get values for xi     ----
xis           = np.arange(xi_min,xi_max+STEP,STEP)
#----   Save info in a tuple    ----
fixed_args    = (molecules,nus,n_0,xis,refdata)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.00,description=r'T [K]'  , **args)
P_slider = w.FloatSlider(value=  1.00,min=  0.01,max=  3.00,step=0.01,description=r'p [bar]', **args)
ui       = w.VBox([T_slider,P_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"TP"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, fixed_args: plot_DG_TP(T, P*1E5, fixed_args), {'T': T_slider, 'P': P_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)


#####
As you have observed, the equilibrium extent (and its corresponding Gibbs free energy) depends on the experimental temperature and pressure. This dependence can be visualized using a 3D plot, which enables us to examine (a) the equilibrium position, $\xi_{\rm eq}$, and (b) the change in Gibbs free energy from the initial state ($\xi = 0$) to the equilibrium state ($\xi_{\rm eq}$).

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
Ts  = np.linspace(200.15,400.15 ,201)
Ps  = np.linspace(0.01E5, 3.01E5,201)
P,T = np.meshgrid(Ps, Ts)
#------------------------------------------------------
plot3D_eqPT_N2O4(T,P)
#------------------------------------------------------

#####

Beyond the previous global view, it is also useful to analyze the equilibrium condition directly in terms of mixture composition through the molar Gibbs free energy, defined as

$$
G_m(\xi) =  G(\xi) / n(\xi) \tag{4}
$$

where both the total Gibbs free energy, $G$, and the total number of moles, $n$, depend on the extent of reaction.

For the binary mixture considered here, this can also be written as:

$$
G_m(\xi) =  y_{N_2O_4} \cdot \mu_{N_2O_4}  + y_{NO_2} \cdot \mu_{NO_2} \tag{5}
$$

By plotting $G_m$ as a function of $y_{NO_2}$, the equilibrium composition can also be identified. Notably, the equilibrium point does not coincide with the minimum of this curve. Execute the following cell and use the $y_{NO_2}$ slider to explore how the equilibrium point can be inferred from the $G_m$ plot (hint: it is related to the _intercept method_).

In [ ]:
#@title <small><small> { display-mode: "form" }

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True)
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.0000  ,description=r'T [K]'  ,readout_format='.2f',**args)
P_slider = w.FloatSlider(value=  1.00,min=  0.01,max=  3.00,step=0.0100  ,description=r'p [bar]',readout_format='.2f',**args)
yB_slider= w.FloatSlider(value=  0.50,min=  0.00,max=  1.00,step=0.000001,description=r'y(NO2)' ,readout_format='.6f',**args)

# --- Separator text between slider groups ---
separator = w.HTML(
    value=(
        "<div style='width:600px; margin:10px 0 6px 0;'>"
        "  <hr style='border:0; border-top:1px solid rgba(255,255,255,0.25); margin:0 0 8px 0;'>"
        "  <div style='text-align:center; font-size:13px; line-height:1.25; opacity:0.85;'>"
        "    Move <b>y(NO2)</b> to select a point on <b>G<sub>m</sub></b>.<br>"
        "    The <b>tangent</b> to <b>G<sub>m</sub></b> will be shown."
        "  </div>"
        "</div>"
    )
)

# sliders
ui = w.VBox([T_slider, P_slider,separator,yB_slider],
            layout=w.Layout(width='600px',display='flex',flex_flow='column',align_items='flex-start',justify_content='flex-start'))

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"TP"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, yB: interceptN2O4_plot(T,P*1E5,yB), {'T': T_slider, 'P': P_slider, 'yB': yB_slider})
display(w.VBox([ui, btn]), out)


####
(a.3) REACTION CONDUCTED AT CONSTANT V AND T

#####
Another case of interest is a reaction carried out in a reactor at constant $T$ and $V$. Under these conditions, the relevant thermodynamic potential is the Helmholtz free energy (A):

$$
A = G - pV = G - nRT
\tag{6}
$$

As in the previous case, let us assume that the reaction vessel initially contains 1 mol of N$_2$O$_4$ (and none of NO$_2$). Execute the cell below, set the temperature and the total volume, and analyze how $A$ changes with $\xi$:


In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
refdata,molecules,nus,n_0        = load_n2o4_2no2()
#------------------------------------------------------

#----  Get limit values for xi  ----
STEP          = 1E-4
xi_min,xi_max = limits_xi(n_0,nus)
#----     Get values for xi     ----
xis           = np.arange(xi_min,xi_max+STEP,STEP)
#----   Save info in a tuple    ----
fixed_args    = (molecules,nus,n_0,xis,refdata)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.00,description=r'T [K]', **args)
V_slider = w.FloatSlider(value= 24.78,min=  2.00,max=250.00,step=0.01,description=r'V [L]', **args)
ui       = w.VBox([T_slider,V_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"TV"))

# -------- slider ---> function --------
# V is converted from L to m3 with *1E-3
out = w.interactive_output(lambda T, V, fixed_args: plot_DA_TV(T, V*1E-3, fixed_args), {'T': T_slider, 'V': V_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

### **(b) Study your own reaction**



#### 1st step: _define your reaction of interest_


In [ ]:
#@title <small><small> { display-mode: "form" }

TEXT1 = '''
===================
Firstly, introduce your reaction, using the following format:

      A + 2 B -> 3 C + 2 D

Make sure to include blank spaces between the stoichiometric coefficients and the chemical species.
For example, for the reaction we studied above, you should enter:

     N2O4 -> 2 NO2
===================

'''

TEXT2 = '''
~~~~~~~~~~~~~~~~~~~
Information:")

 * reactants (%i): %s
 * products  (%i): %s

 * equation for the reaction: %s
~~~~~~~~~~~~~~~~~~~

'''

# ---- Functions for getting the user reaction ----
def ask_for_reaction(max_nerr=3):

    nus0,molecules0 = None,None

    print(TEXT1)
    answer,nerr = "n",0
    while True:
        if nerr == max_nerr:
            print("-------------------")
            print("Too many errors... aborting...")
            print("-------------------")
            return nus0,molecules0
        try:
            reaction = input(" * insert reaction: ")
            print("")
            nus,molecules = string_to_reaction(reaction)
            if nus is None: raise Exception

            #---- Print info to make sure all is correct ----
            string = reaction_to_string(nus,molecules)
            nR = len([nu for nu in nus if nu<0])
            nP = len([nu for nu in nus if nu>0])
            sR = ", ".join([molecule for nu,molecule in zip(nus,molecules) if nu < 0])
            sP = ", ".join([molecule for nu,molecule in zip(nus,molecules) if nu > 0])
            print(TEXT2%(nR,sR,nP,sP,string))
            answer = input("Is the reaction correct? (Type 'yes' or 'no'): ")
            answer = answer.strip().lower()[0]
            if answer == "y":
               print("\nGreat! Now, execute the next cell!\n\n")
               return nus,molecules
            else:
               print("Ok, so let us try it again! :)\n\n")
               nerr += 1
        except:
            nerr += 1
            print("-------------------")
            print("There was some kind of problem... Let us try again!")
# ------------------------------------------------

# ---- Ask for reaction ----
nus, molecules = ask_for_reaction(max_nerr=3)


#### 2nd step: _introduce the thermodynamical data_

In [ ]:
#@title <small><small> { display-mode: "form" }

magnitudes = []
magnitudes += [("H","formation entalphy [kJ/mol]")]
magnitudes += [("S","entropy [J/(mol K)]")]
magnitudes += [("C","molar heat capacities at constant P [J/(mol K)]")]

print(rf"We need the thermodynamical data for all the species.")

# ---- Get thermodynamical data ----
T_ref = ask_for_float("First, set the reference temperature (K): ")
print("")
thermodata = {}
for molecule in molecules:
    print(rf"Introduce the thermodynamical data for species: {molecule:s}")
    print("")
    thermodata[molecule] = {}
    for key,magnitude in magnitudes:
        count = 0
        thermodata[molecule][key] = ask_for_float(rf"    * {magnitude:47s}: ")
    print("")

# ---- Get values for the reaction ----
DHo  = sum([nu_i*thermodata[molec_i]["H"] for nu_i, molec_i in zip(nus,molecules)])
DSo  = sum([nu_i*thermodata[molec_i]["S"] for nu_i, molec_i in zip(nus,molecules)])
DCPo = sum([nu_i*thermodata[molec_i]["C"] for nu_i, molec_i in zip(nus,molecules)])

# ---- Print values for the reaction ----
print(rf"Magnitudes of interest at {T_ref:.2f} K:")
print(rf"    Delta_r{{H}}^o  = {DHo:+8.2f} kJ/mol")
print(rf"    Delta_r{{S}}^o  = {DSo:+8.2f}  J/(mol K)")
print(rf"    Delta_r{{Cp}}^o = {DCPo:+8.2f}  J/(mol K)")

# ---- Convert Entalphy of reaction to SI ----
DHo *= 1000

# ---- Save data ----
refdata = (DHo,DSo,DCPo,T_ref)
#------------------------------------------------------

#### 3rd step: _examine the effect of temperature on $\Delta_{r}{G}^\circ(T)$_

In [ ]:
#@title <small><small> { display-mode: "form" }

# min and max value for temperature
deltaT    = 200
Tmin,Tmax = max(T_ref-deltaT,0), T_ref+deltaT
# plot data
T     = np.linspace(Tmin,Tmax,10)
DGo_T = plot_DG_T(T,T_ref,refdata)


#### 4th step: _analyze the equilibrium under different conditions_


##### (4.a): _set the initial number of moles_


In [ ]:
#@title <small><small> { display-mode: "form" }

print("Let us set the initial conditions")
print("")

print("   - indicate, first, the number of moles of each species!")
#-----------------------------------
n_0 = []
for molecule in molecules:
    nmol = ask_for_float(rf"     n_mol({molecule:s}): ")
    n_0.append(nmol)
n_0 = np.array(n_0)
#-----------------------------------
STEP          = 1E-4
xi_min,xi_max = limits_xi(n_0,nus)
xis           = np.arange(xi_min,xi_max+STEP,STEP)
fixed_args    = (molecules,nus,n_0,xis,refdata)
#-----------------------------------
print("")
print(rf"   min value for xi: {xi_min:+7.3f} mol")
print(rf"   max value for xi: {xi_max:+7.3f} mol")
print("")

##### (4.b): _equilibrium at constant T and P_

In [ ]:
#@title <small><small> { display-mode: "form" }

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=T_ref,min=Tmin,max=Tmax,step=1.00,description=r'T [K]'  , **args)
P_slider = w.FloatSlider(value= 1.00,min=0.01,max=5.00,step=0.01,description=r'p [bar]', **args)
ui       = w.VBox([T_slider,P_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"TP"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, fixed_args: plot_DG_TP(T, P*1E5, fixed_args), {'T': T_slider, 'P': P_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

##### (4.c): _equilibrium at constant T and V_

In [ ]:
#@title <small><small> { display-mode: "form" }

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=T_ref,min=Tmin,max=Tmax  ,step=1.00,description=r'T [K]', **args)
V_slider = w.FloatSlider(value=24.80,min=2.00,max=500.00,step=0.20,description=r'V [L]', **args)
ui       = w.VBox([T_slider,V_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"TV"))

# -------- slider ---> function --------
# V is converted from L to m3 with *1E-3
out = w.interactive_output(lambda T, V, fixed_args: plot_DA_TV(T, V*1E-3, fixed_args), {'T': T_slider, 'V': V_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)